[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dbamman/anlp25/blob/main/3.embeddings/Lexical_Semantics.ipynb)

**N.B.** Once it's open on Colab, remember to save a copy (by e.g. clicking `Copy to Drive` above).

---

## Homework 3: Lexical Semantics

In this homework, we will explore lexical semantics in the context of slang and FastText, an alternative to word2vec (Part 1); and, how to represent a sentence with individual word vectors, so we can measure the similarity between a pair of sentences (Part 2).

### Part 1: Slang and word similarity with FastText

Slang presents an interesting linguistic phenomenon that involves non-standard word forms. For this question, you will explore how FastText, an alternative to Word2Vec, handles the lexical semantics of slang and informal language.

First, **familiarize yourself with the slang dataset that we are using**, introduced in ["Toward Informal Language Processing: Knowledge of Slang in Large Language Models" (Sun et al., NAACL 2024)](https://aclanthology.org/2024.naacl-long.94/). The full dataset includes annotations indicating whether a sentence from OpenSubtitles (typically a line from a movie) contains a slang term, and you can find some example sentences and terms here:

https://raw.githubusercontent.com/dbamman/anlp25/main/data/slang_examples.tsv

In [65]:
# Import libraries NLTK and PDF plumber
# download the packages first here
!pip install nltk
!pip install PyPDF2
!pip install gensim
import nltk
nltk.download('punkt_tab')
import requests # Download from github
from PyPDF2 import PdfReader # PDF reader
from io import BytesIO
import math
import operator
from collections import Counter

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


What are the 10 most common slang words?

In [66]:
!wget --no-check-certificate https://raw.githubusercontent.com/dbamman/anlp25/main/data/slang_examples.tsv // Download the dataset



--2025-09-16 02:01:31--  https://raw.githubusercontent.com/dbamman/anlp25/main/data/slang_examples.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 376405 (368K) [text/plain]
Saving to: ‘slang_examples.tsv.35’

slang_examples.tsv. 100%[===================>] 367.58K  --.-KB/s    in 0.05s   

2025-09-16 02:01:31 (7.69 MB/s) - ‘slang_examples.tsv.35’ saved [376405/376405]

//: Scheme missing.
--2025-09-16 02:01:31--  http://download/
Resolving download (download)... failed: No address associated with hostname.
wget: unable to resolve host address ‘download’
--2025-09-16 02:01:31--  http://the/
Resolving the (the)... failed: Name or service not known.
wget: unable to resolve host address ‘the’
--2025-09-16 02:01:31--  http://dataset/
Resolving dataset (datase

In [67]:

# Read the TSV file
import pandas as pd
df = pd.read_csv('slang_examples.tsv', sep='\t', header=0, names=['SENTENCE', 'SLANG_TERM'])
# get top 10 Slang terms
print (" These are the top 10 slang terms", df['SLANG_TERM'].value_counts().head(10))


 These are the top 10 slang terms SLANG_TERM
gonna    407
yeah     182
shit     117
wanna    105
ain't     76
mate      63
gotta     63
man       47
okay      47
kid       46
Name: count, dtype: int64


Next, **train a [FastText model](https://radimrehurek.com/gensim/models/fasttext.html#gensim.models.fasttext.FastText) using `gensim` and `FastText`** on our slang data derived from the dataset described above, which you can download here:

https://raw.githubusercontent.com/dbamman/anlp25/main/data/slang_corpus.txt

For preprocessing, in the `txt` file, each token is already separated by whitespace, so you don't need to worry about tokenization. Treat each line in the file as one "sentence". For training, use the following parameters: embedding size of 400, context window of 5, frequency threshold of 5, and use 5 workers to train for 5 epochs.

Note: we are using the `FastText` _implementation_ included in the `gensim` library! **Don't use the `fasttext` library.**

In [68]:
import re
!wget --no-check-certificate https://raw.githubusercontent.com/dbamman/anlp25/main/data/slang_corpus.txt // Download the dataset
file= open("slang_corpus.txt", "r")
text = file.read()
words_list = []
# use NLTK word tokenize for tokenizing
for sentence in text.split("\n"):
  words=re.sub("\s+", " ", sentence.rstrip().lower()) # Removing white space
  words_list.append(words.split(" "))

print(words_list)
# would be good


--2025-09-16 02:01:31--  https://raw.githubusercontent.com/dbamman/anlp25/main/data/slang_corpus.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 

<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-3939882400.py:9: SyntaxWarning: invalid escape sequence '\s'
  words=re.sub("\s+", " ", sentence.rstrip().lower())


200 OK
Length: 2868710 (2.7M) [text/plain]
Saving to: ‘slang_corpus.txt.9’

slang_corpus.txt.9  100%[===================>]   2.74M  --.-KB/s    in 0.08s   

2025-09-16 02:01:31 (33.9 MB/s) - ‘slang_corpus.txt.9’ saved [2868710/2868710]

//: Scheme missing.
--2025-09-16 02:01:31--  http://download/
Resolving download (download)... failed: No address associated with hostname.
wget: unable to resolve host address ‘download’
--2025-09-16 02:01:31--  http://the/
Resolving the (the)... failed: Name or service not known.
wget: unable to resolve host address ‘the’
--2025-09-16 02:01:31--  http://dataset/
Resolving dataset (dataset)... failed: Name or service not known.
wget: unable to resolve host address ‘dataset’
FINISHED --2025-09-16 02:01:31--
Total wall clock time: 0.3s
Downloaded: 1 files, 2.7M in 0.08s (33.9 MB/s)
[['because', 'that', 'would', 'make', 'things', 'even', 'better', '.', 'or', 'maybe', 'we', 'could', 'go', 'somewhere', 'and', 'you', 'could', 'burn', 'your', 'tongue', 'on', 

In [69]:
# plug the words in gensim library train
from gensim.models import FastText
model = FastText(sentences=words_list, vector_size=400, window=5, min_count=5, workers=5, epochs=5)

With the trained models:

**Q1.** Pick a slang term, and in about 100 words, discuss:

- what the most similar words are to the slang term of your choosing (as measured by the model), and
- whether the result is aligned with your understanding.

In [77]:

word_embedding = model.wv['shit']

# Most similar words to a given word
similar_words = model.wv.most_similar('shit',topn=10)
print(similar_words)

[('kit', 0.8970131278038025), ('ship', 0.8754363059997559), ('jody', 0.8654780983924866), ('bloody', 0.8577790260314941), ('bait', 0.8523259162902832), ('bullshit', 0.8492954969406128), ('carry', 0.8490222096443176), ('shh', 0.8431004881858826), ('fucker', 0.8379709124565125), ('hell', 0.8375301957130432)]




---
Looking at the results, it seems like the model was able to includes words like 'hell', 'bloody', 'shh' which communicates strong emotion like 'shit'. I would expect these words to be around the same vector position.  Some words like 'kit', 'ship' seems to come from spelling similarity rather than the cosine similarity we know of.


---


The results aligns slightly with my understanding but I did not expect to see words such as 'Ship' 'Carry' and 'kit'. Probably this could be due to some lingustic context that is within the text.

**Q2.** Train a separate word2vec (not FastText) model using the same dataset, and in about 100 words, compare the two approaches used to estimate word vectors. Here are some potential topics:

- Look up the token `gonna` in both word2vec and FastText models. What does this tell you?
- What is the high level difference between word2vec and FastText?
- Evaluate the quality of the trained embeddings through intrinsic evaluation.

In [79]:
from gensim.models import Word2Vec, KeyedVectors

model2 = Word2Vec(
        sentences=words_list,
        vector_size=100,
        window=5,
        min_count=2,
        workers=10)
word_embedding2 = model2.wv['shit']

# Most similar words to a given word
similar_words = model2.wv.most_similar('shit', topn=10)
print(similar_words)

[('fucking', 0.8257643580436707), ('-', 0.7963502407073975), ('blood', 0.7961048483848572), ('fuck', 0.7935729026794434), ('ok', 0.7795041799545288), ('mate', 0.7728798389434814), ('gun', 0.7656990885734558), ('okay', 0.7615877985954285), ('yo', 0.7587260007858276), ('allright', 0.7512235641479492)]




---

The words produced by word2vec seems to be closely aligned or semantically coherent with the word 'shit'.  It does show a strong performance since these words have same contextual/semantic meanings



---



In [100]:
# Model evalutaion using a common wordsim353.tsv
from gensim.test.utils import datapath

model.wv.evaluate_word_pairs(datapath('wordsim353.tsv'))


(PearsonRResult(statistic=-0.037758526193288136, pvalue=0.6218632113528049),
 SignificanceResult(statistic=-0.013157330549032996, pvalue=0.8635865056589429),
 50.991501416430594)

In [88]:
model2.wv.evaluate_word_pairs(datapath('wordsim353.tsv'))

(PearsonRResult(statistic=0.08145305206103114, pvalue=0.2048295096430905),
 SignificanceResult(statistic=0.08784213575740826, pvalue=0.17139431500050606),
 30.878186968838527)

- Seems model 2 , the Word2vec model is strong compared to fastText model since model 2 has lower p-value compared to the other one.

### Part 2. From words to sentences

So far we've been working with word vectors, but in real-world scenarios, we often want to work with not just a word, but a sequence (like a sentence), which will explore later in the semester. However, with what we have learned so far, how do you represent a sentence? One approach is to look up the word vectors for individual words in the sentence and then *average* them, which we will explore in this question. We will be using pre-trained GloVe vectors [cf. SLP 6.8.3] we used in class. Download them here:

https://raw.githubusercontent.com/dbamman/anlp25/main/data/glove.6B.100d.100K.txt

**Q3.** Load the pretrained embeddings with `gensim`'s `load_word2vec_format` (see the lab notebooks), and create a function that takes a pair of sentences as input, and outputs the similarity of the two sentences measured by cosine -- the sentence pair you can use for sanity check is provided below.

Find a pair of sentences where the similarity is high, but mean different (or opposite) things. Find a pair of sentences where the similarity is low, but you think the meanings are similar. In a paragraph, discuss why we might see these results given how we construct sentence embeddings.

In [72]:
# sanity check: compare the similarity between:
#       sentences[0], sentences[1]
#       sentences[0], sentences[2]

sentences = [
    ['the', 'cat', 'is', 'on', 'the', 'mat'],
    ['a', 'computer', 'is', 'in', 'the', 'store'],
    ['the', 'cat', 'sits', 'on', 'the', 'mat']
]

----

## To submit

Congratulations on finishing this homework!
Please follow the instructions below to download the notebook file (`.ipynb`) and its printed version (`.pdf`) for submission on bCourses -- remember **all cells must be executed**.